<a href="https://colab.research.google.com/github/FernandaChacara/AVCAD/blob/main/Exercise5/Exercise_5_Dandara_Fernanda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

# Isto irá abrir uma janela para selecionar e carregar o seu ficheiro
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'Ficheiro "{fn}" carregado com sucesso!')
  # Renomear o ficheiro carregado para o nome esperado se for diferente
  if fn != 'EFIplus_medit.csv':
      %mv "{fn}" "/content/EFIplus_medit.csv"


Saving EFIplus_medit.csv to EFIplus_medit (1).csv
Ficheiro "EFIplus_medit (1).csv" carregado com sucesso!


In [ ]:
print(df.columns.tolist())

['Site_code', 'Latitude', 'Longitude', 'Country', 'Catchment_name', 'Galiza', 'Subsample', 'Calib_EFI_Medit', 'Calib_connect', 'Calib_hydrol', 'Calib_morphol', 'Calib_wqual', 'Geomorph1', 'Geomorph2', 'Geomorph3', 'Water_source_type', 'Flow_regime', 'Altitude', 'Geological_typology', 'Actual_river_slope', 'Natural_sediment', 'Elevation_mean_catch', 'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul', 'Barriers_catchment_down', 'Barriers_river_segment_up', 'Barriers_river_segment_down', 'Barriers_number_river_segment_up', 'Barriers_number_river_segment_down', 'Barriers_distance_river_segment_up', 'Barriers_distance_river_segment_down', 'Impoundment', 'Hydropeaking', 'Water_abstraction', 'Hydro_mod', 'Temperature_impact', 'Velocity_increase', 'Reservoir_flushing', 'Sedimentation', 'Channelisation', 'Cross_sec', 'Instream_habitat', 'Riparian_vegetation', 'Embankment', 'Floodprotection', 'Floodplain', 'Toxic_substances', 'Acidification', 'Water_quality_index', 'Eutrophication', 'Organic_p

In [ ]:
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler

# carregar dados
df = pd.read_csv("/content/EFIplus_medit.csv", sep=';')

# filtrar espécie
trout = df[df["Salmo trutta fario"].notna()]

# grupos
presence = trout[trout["Salmo trutta fario"] == 1]["temp_ann"].dropna()
absence = trout[trout["Salmo trutta fario"] == 0]["temp_ann"].dropna()

# --- TESTE NÃO PARAMÉTRICO ---
u_stat, p_value = stats.mannwhitneyu(presence, absence)

print("Mann-Whitney p-value:", p_value)

# --- PADRONIZAÇÃO ---
scaler = StandardScaler()
df["temp_ann_std"] = scaler.fit_transform(df[["temp_ann"]])

# Recriar presence_std e absence_std com o nome da coluna corrigido e remover NaNs
presence_std = df[df["Salmo trutta fario"] == 1]["temp_ann_std"].dropna()
absence_std = df[df["Salmo trutta fario"] == 0]["temp_ann_std"].dropna()

u_stat_std, p_value_std = stats.mannwhitneyu(presence_std, absence_std)

print("Standardized p-value:", p_value_std)

Mann-Whitney p-value: 7.105075261935899e-303
Standardized p-value: 7.105075261935899e-303


In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

# tabela de contingência
cont_table = pd.crosstab(df["Country"], df["Salmo trutta fario"])

chi2, p, dof, expected = chi2_contingency(cont_table)

print("p-value:", p)

p-value: 2.9162328651936495e-107


In [ ]:
import plotly.express as px

fig = px.parallel_categories(
    df,
    dimensions=["Country", "Salmo trutta fario"]
)

fig.show()

In [ ]:
import numpy as np
from scipy.stats import f_oneway

# top 8 catchments
top8 = df["Catchment_name"].value_counts().head(8).index
df_top = df[df["Catchment_name"].isin(top8)]

# agrupar dados e remover NaNs
groups = [df_top[df_top["Catchment_name"] == c]["Elevation_mean_catch"].dropna() for c in top8]

# ANOVA
f_stat, p_value = f_oneway(*groups)

print("ANOVA p-value:", p_value)

ANOVA p-value: 1.3695264820354472e-285


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Filtrar df_top para remover NaNs na coluna Elevation_mean_catch para o teste de Tukey
df_tukey = df_top.dropna(subset=["Elevation_mean_catch"])

tukey = pairwise_tukeyhsd(
    df_tukey["Elevation_mean_catch"],
    df_tukey["Catchment_name"]
)

print(tukey)

         Multiple Comparison of Means - Tukey HSD, FWER=0.05         
   group1       group2     meandiff p-adj    lower     upper   reject
---------------------------------------------------------------------
  Cantabrica       Catala   50.1883 0.7208  -42.1585  142.5351  False
  Cantabrica        Douro  268.1584    0.0  190.9443  345.3726   True
  Cantabrica         Ebro  467.4299    0.0   399.797  535.0628   True
  Cantabrica Galiza-Norte -184.2794    0.0   -252.02 -116.5388   True
  Cantabrica       Guadia -168.8947    0.0 -251.7428  -86.0466   True
  Cantabrica        Minho  290.9895    0.0  223.2126  358.7663   True
  Cantabrica         Tejo  168.3227    0.0    95.179  241.4664   True
      Catala        Douro  217.9701    0.0  124.2303  311.7099   True
      Catala         Ebro  417.2415    0.0  331.2221  503.2609   True
      Catala Galiza-Norte -234.4677    0.0 -320.5718 -148.3636   True
      Catala       Guadia -219.0831    0.0 -317.5154 -120.6507   True
      Catala        

In [ ]:
from scipy.stats import kruskal

h_stat, p_value = kruskal(*groups)

print("Kruskal p-value:", p_value)

Kruskal p-value: 3.7056116510329714e-284


In [ ]:
!pip install scikit-posthocs

import scikit_posthocs as sp

posthoc = sp.posthoc_dunn(df_top, val_col="Elevation_mean_catch", group_col="Catchment_name")

print(posthoc)

                Cantabrica        Catala         Douro           Ebro  \
Cantabrica    1.000000e+00  3.366175e-02  6.495985e-31   1.352663e-65   
Catala        3.366175e-02  1.000000e+00  1.083257e-13   5.795173e-29   
Douro         6.495985e-31  1.083257e-13  1.000000e+00   1.444488e-04   
Ebro          1.352663e-65  5.795173e-29  1.444488e-04   1.000000e+00   
Galiza-Norte  3.227136e-14  1.604135e-16  9.056037e-91  1.684344e-177   
Guadia        3.459856e-08  3.227288e-11  1.452004e-57  8.960435e-101   
Minho         1.569986e-34  1.753349e-13  3.732121e-01   2.984973e-08   
Tejo          1.316758e-09  6.196547e-03  2.028710e-09   4.891813e-28   

               Galiza-Norte         Guadia          Minho          Tejo  
Cantabrica     3.227136e-14   3.459856e-08   1.569986e-34  1.316758e-09  
Catala         1.604135e-16   3.227288e-11   1.753349e-13  6.196547e-03  
Douro          9.056037e-91   1.452004e-57   3.732121e-01  2.028710e-09  
Ebro          1.684344e-177  8.960435e-101   2